## PCA por Métricas

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

from scipy.stats import ttest_ind

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"

### Carregando o dataset

In [2]:
DATASET_PATH = "../../data/processed/dataset_features.parquet"

df_dataset = pd.read_parquet(DATASET_PATH)
df_dataset.shape

(108, 360)

In [3]:
#  Age, Gender and condition
# Create a dictionary where keys are the metric names (e.g., 'degree')
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "hub_frequency_",
]

NODE_FEATURES = [
    c for c in df_dataset.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]

NODE_FEATURES_BY_TYPE = {
    prefix.rstrip("_"): [c for c in NODE_FEATURES if c.startswith(prefix)]
    for prefix in NODE_FEATURE_PREFIXES
}

NODE_GROUPS = {
    "degree": [c for c in df_dataset.columns if c.startswith("degree_")],
    "clustering": [c for c in df_dataset.columns if c.startswith("clustering_")],
    "betweenness": [c for c in df_dataset.columns if c.startswith("betweenness_")],
}

In [4]:
def plot_pca_features(df, features, title="PCA"):
    # valor ausente = eletrodo ausente no participante, não conectividade zero
    X = SimpleImputer(strategy="median").fit_transform(df[features])
    X_scaled = StandardScaler().fit_transform(X)

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
    if "condition" in df.columns:
        df_pca["condition"] = df["condition"].values
        color_col = "condition"
    else:
        color_col = None

    fig = px.scatter(
        df_pca, x="PC1", y="PC2",
        color=color_col,
        title=title,
        opacity=0.8
    )
    fig.show()

    ratio = pca.explained_variance_ratio_
    loadings = pd.DataFrame(pca.components_.T, index=features, columns=["PC1", "PC2"])
    same_sign = max((loadings["PC1"] > 0).mean(), (loadings["PC1"] < 0).mean())
    print(f"Variância explicada: PC1 = {ratio[0]:.1%} | PC2 = {ratio[1]:.1%}")
    print(f"Cargas de PC1 com o mesmo sinal: {same_sign:.0%} dos eletrodos")
    display(loadings.reindex(loadings["PC1"].abs().sort_values(ascending=False).index).head(10).round(3))
    return pca, loadings

Grau

In [5]:
pca_degree, loadings_degree = plot_pca_features(
    df_dataset,
    NODE_FEATURES_BY_TYPE["degree"],
    title="PCA das Métricas de Grau (Nós)"
)

Variância explicada: PC1 = 82.0% | PC2 = 4.8%
Cargas de PC1 com o mesmo sinal: 100% dos eletrodos


,PC1,PC2
degree_P3,0.146,-0.098
degree_P5,0.145,-0.109
degree_P6,0.145,-0.092
degree_P4,0.145,-0.123
degree_PO4,0.144,-0.107
degree_Oz,0.144,-0.057
degree_P8,0.143,-0.034
degree_CP6,0.143,-0.029
degree_PO3,0.143,-0.120
degree_P7,0.143,-0.022


Clustering

In [6]:
pca_clustering, loadings_clustering = plot_pca_features(
    df_dataset,
    NODE_FEATURES_BY_TYPE["clustering"],
    title="PCA das Métricas de Clustering"
)

Variância explicada: PC1 = 91.1% | PC2 = 2.1%
Cargas de PC1 com o mesmo sinal: 100% dos eletrodos


,PC1,PC2
clustering_P3,0.142,-0.090
clustering_P6,0.142,-0.084
clustering_P4,0.142,-0.110
clustering_P5,0.141,-0.102
clustering_PO4,0.141,-0.095
clustering_Oz,0.141,-0.048
clustering_PO3,0.141,-0.104
clustering_P8,0.141,-0.037
clustering_PO7,0.140,-0.050
clustering_P7,0.140,-0.028


Centralidade de Intermediação

In [7]:
pca_betweenness, loadings_betweenness = plot_pca_features(
    df_dataset,
    NODE_GROUPS["betweenness"],
    title="PCA da Centralidade de Intermediação (Nós)"
)

Variância explicada: PC1 = 23.5% | PC2 = 9.1%
Cargas de PC1 com o mesmo sinal: 100% dos eletrodos


,PC1,PC2
betweenness_C3,0.226,-0.140
betweenness_CP1,0.224,-0.150
betweenness_C1,0.222,-0.183
betweenness_CP4,0.212,-0.170
betweenness_C4,0.204,-0.097
betweenness_CP3,0.201,-0.162
betweenness_FC2,0.201,-0.123
betweenness_P1,0.198,-0.150
betweenness_P5,0.196,0.093
betweenness_P2,0.195,-0.159
